## Path configuration

In [ ]:
from pathlib import Path
import os

PROJECT_NAME = "MALDIAlign"

cwd = Path().resolve()

# Walk upwards until we find the project folder
target = None
for parent in [cwd] + list(cwd.parents):
    if parent.name == PROJECT_NAME:
        target = parent
        break

# If the project folder is found and we are not already there, then change cwd
if target is not None and target != cwd:
    os.chdir(target)

print("Working directory:", os.getcwd())

## Imports

In [ ]:
import numpy as np
import pandas as pd

from src.config.loader import load_config
from src.data.io import verify_data_path, load_pkl

## Data loading

In [2]:
# Obtain the path from the dataset that we want to work with
cfg = load_config()
marisma_pkl = cfg["data"]["MARISMa_REDUCED_PKL"]

In [3]:
# Verify that it exists
verify_data_path(marisma_pkl)

Path exists: /export/data_ml4ds/bacteria_id/codigoMALDIVAS/pickles/MARISMa_study.pkl


In [4]:
# Load MARISMa pickle file
marisma = load_pkl(marisma_pkl)

## Data exploration

In [5]:
print(type(marisma))
print(marisma.keys())

<class 'dict'>
dict_keys(['data', 'label', 'meta'])


In [6]:
data, label, meta = marisma["data"], marisma["label"], marisma["meta"]

In [7]:
print(f"Spectra matrix shape: {data.shape}")
print(f"Labels shape: {label.shape}")
print(f"Metadata shape: {meta.shape}")

Spectra matrix shape: (76172, 6000)
Labels shape: (76172,)
Metadata shape: (76172,)


In [8]:
print(type(data))
print(type(label))
print(type(meta))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


Take a closer look at each variable to familiarize with its structure

In [9]:
print(data[:5])

[[0.         0.         0.54512112 ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


The variable `data` is a 2D NumPy array of shape (n_samples, n_features).
- Rows: Individual spectra
    - Each row corresponds to one MALDI-TOF spectrum
    - That is, one bacterial isolate (sample). For example, data[0] might represent a Pseudomonas aeruginosa isolate measured in 2020.

- Columns: m/z bins
    - Each column represents an intensity value at a specific mass-to-charge (m/z) bin.
    - During preprocessing, the continuous m/z range was discretized into fixed bins (e.g., 6 000). So column j = intensity measured around a certain m/z interval.

- Values → Normalized signal intensities
    - Each number (e.g., 0.54512112) is the normalized ion intensity at that m/z bin:
        - 0.0 → no signal (flat region)
        - 0.5 → moderate signal
        - 1.0 → highest peak in that spectrum

**Summary table**

| Level        | Meaning                          | Example                         |
| ------------ | -------------------------------- | ------------------------------- |
| Row `i`      | One spectrum / bacterial isolate | *Pseudomonas aeruginosa* (2020) |
| Column `j`   | One m/z interval (binned)        | Signal around 3200 Da           |
| `data[i, j]` | Normalized intensity (0–1)       | 0.545 → medium peak strength    |


In [10]:
print(label[:5])

['Pseudomonas_Aeruginosa' 'Pseudomonas_Aeruginosa'
 'Pseudomonas_Aeruginosa' 'Pseudomonas_Aeruginosa'
 'Pseudomonas_Aeruginosa']


In [11]:
print(np.unique(label))

['Enterobacter_cloacae_complex' 'Enterococcus_Faecium' 'Escherichia_Coli'
 'Klebsiella_Pneumoniae' 'Pseudomonas_Aeruginosa' 'Staphylococcus_Aureus']


`label` is a 1D array with one entry per spectrum in data.
- Each element represents the bacterial species corresponding to that spectrum.
- Example: label[0] = "Pseudomonas_Aeruginosa" → data[0] is a spectrum from P. aeruginosa.

The dataset contains six distinct species:
- Enterobacter cloacae complex
- Enterococcus faecium
- Escherichia coli
- Klebsiella pneumoniae
- Pseudomonas aeruginosa
- Staphylococcus aureus

label[i] aligns directly with data[i] and meta[i] → all three describe the same sample.

**Summary table**
| Level                     | Meaning                                       | Example                                        |
| ------------------------- | --------------------------------------------- | ---------------------------------------------- |
| `label[i]`                | Species name of sample *i*                    | `"Pseudomonas_Aeruginosa"`                     |
| `np.unique(label)`        | All species present in the dataset            | 6 species (listed above)                       |
| Relationship to `data[i]` | Defines the class / species for that spectrum | Used for supervised training or coloring plots |

In [12]:
print(meta)
marisma["meta"][10]

[{'year': '2020', 'genus': 'Pseudomonas', 'species': 'Aeruginosa', 'study': 'ac9d3149/0_C2/1'}
 {'year': '2020', 'genus': 'Pseudomonas', 'species': 'Aeruginosa', 'study': '840724c3/0_D7/1'}
 {'year': '2020', 'genus': 'Pseudomonas', 'species': 'Aeruginosa', 'study': '840724c3/0_C5/1'}
 ...
 {'year': '2022', 'genus': 'Enterobacter', 'species': 'Kobei', 'study': '57e2ec67-2/0_A5/1'}
 {'year': '2022', 'genus': 'Enterobacter', 'species': 'Kobei', 'study': '6ae3dc2d-2E/0_B10/1'}
 {'year': '2022', 'genus': 'Enterobacter', 'species': 'Kobei', 'study': '2c6f70a3/0_A8/1'}]


{'year': '2020',
 'genus': 'Pseudomonas',
 'species': 'Aeruginosa',
 'study': '14a578ca/0_A5/1'}

`meta` contains metadata dictionaries describing each sample (one per spectrum).
- Each row corresponds to the same sample index as data[i] and label[i].
- In the pickle, it is stored as a DataFrame with a single column of dictionaries.

**Summary table**
| Key       | Meaning                                                | Example             |
| --------- | ------------------------------------------------------ | ------------------- |
| `year`    | Year when the sample was measured                      | `'2020'`            |
| `genus`   | Bacterial genus                                        | `'Pseudomonas'`     |
| `species` | Bacterial species (same as `label`)                    | `'Aeruginosa'`      |
| `study`   | Internal study or sample ID (unique identifier / path) | `'14a578ca/0_A5/1'` |


In [13]:
meta_expanded = pd.DataFrame.from_records(list(meta))

In [14]:
print(meta_expanded.shape)
print(type(meta_expanded))

(76172, 4)
<class 'pandas.core.frame.DataFrame'>


In [15]:
meta_expanded.head()

,year,genus,species,study
0,2020,Pseudomonas,Aeruginosa,ac9d3149/0_C2/1
1,2020,Pseudomonas,Aeruginosa,840724c3/0_D7/1
2,2020,Pseudomonas,Aeruginosa,840724c3/0_C5/1
3,2020,Pseudomonas,Aeruginosa,5db4adbf/0_B11/1
4,2020,Pseudomonas,Aeruginosa,8662f7d6/0_A4/1


In [16]:
for col in meta_expanded.columns:
    print(f"Number of unique labels for column '{col}': {len(np.unique(meta_expanded[col]))}")
    print(f"Unique labels for column '{col}': {np.unique(meta_expanded[col])}", "\n")

Number of unique labels for column 'year': 7
Unique labels for column 'year': ['2018' '2019' '2020' '2021' '2022' '2023' '2024'] 

Number of unique labels for column 'genus': 6
Unique labels for column 'genus': ['Enterobacter' 'Enterococcus' 'Escherichia' 'Klebsiella' 'Pseudomonas'
 'Staphylococcus'] 

Number of unique labels for column 'species': 10
Unique labels for column 'species': ['Aeruginosa' 'Asburiae' 'Aureus' 'Cloacae' 'Coli' 'Faecium' 'Hormaechei'
 'Kobei' 'Ludwigii' 'Pneumoniae'] 

Number of unique labels for column 'study': 76172
Unique labels for column 'study': ['0002adfc/0_A7/1' '0007c050/0_B11/1' '0007de03/0_B2/1' ...
 'fffbb435/0_A1/1' 'fffed254/0_A6/1' 'ffff135c/0_A11/1'] 

